In [61]:
import csv
import uuid
import time
import random
from datetime import datetime, timezone
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import numpy as np
import pandas as pd
class SimulatedUserAgent:
    def __init__(self, age_group, tech_savviness, interests, time_pattern, device):
        self.user_id = str(uuid.uuid4())
        self.age_group = age_group
        self.tech_savviness = tech_savviness
        self.interests = interests
        self.time_pattern = time_pattern
        self.device = device

    def simulate_session(self, session_id=None):
        session_id = session_id or str(uuid.uuid4())
        session_length = self._determine_session_length()
        session_data = []

        for _ in range(session_length):
            action = self._generate_action()
            session_data.append({
                "user_id": self.user_id,
                "session_id": session_id,
                "age_group": self.age_group,
                "tech_savviness": self.tech_savviness,
                "time_pattern": self.time_pattern,
                "device": self.device,
                **action
            })

        return session_data

    def _determine_session_length(self):
        if self.tech_savviness == "Low":
            return max(1, int(np.random.lognormal(mean=1.2, sigma=0.4)))
        elif self.tech_savviness == "Medium":
            return max(1, int(np.random.lognormal(mean=1.6 , sigma=0.5)))
        else:
            return max(1, int(np.random.lognormal(mean=2, sigma=0.6)))
        

    def _generate_action(self):
        actions = ["scroll","click","hover","search"]
        action_count = np.random.poisson(lam={"Low": 3, "Medium": 5, "High": 8}[self.tech_savviness])
        action_type = random.choices(actions, k=1)[0]
        scroll_depth = round(np.random.beta(5,2),2)
        if self.tech_savviness == "High":
            scroll_depth = round(np.random.beta(5,2),2)
        return{
            "actions": action_type,
            "interest": random.choices(self.interests),
            "scroll_depth": scroll_depth,
            "clicks_in_actions": action_count
        }
        
            

def generate_random_agent():
    return SimulatedUserAgent(
        age_group=random.choice(["Teen", "Young Adult", "Adult", "Senior"]),
        tech_savviness=random.choice(["Low", "Medium", "High"]),
        interests=random.sample(
            ["Tech", "Finance", "Gaming", "Health", "Education", "News", "Shopping"], 3
        ),
        time_pattern=random.choice(["Morning", "Afternoon", "Night Owl"]),
        device=random.choice(["Mobile", "Desktop"])
    )
agents = [generate_random_agent() for _ in range(10000)]

all_sessions = []
for agent in agents:
    session = agent.simulate_session()
    all_sessions.extend(session)

class WebBrowsingAgent:
    def __init__(self, age_group, tech_savviness, interests, device):
        self.user_id = str(uuid.uuid4())
        self.age_group = age_group
        self.tech_savviness = tech_savviness
        self.interests = interests
        self.device = device
        self.session_id = str(uuid.uuid4())
        self.logs = []
        self.items_added_to_cart = 0

    def log_action(self, action_type, page_url):
        self.logs.append({
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "user_id": self.user_id,
            "session_id": self.session_id,
            "age_group": self.age_group,
            "tech_savviness": self.tech_savviness,
            "interests": ','.join(self.interests),
            "device": self.device,
            "action": action_type,
            "page_url": page_url,
            "items_added_to_cart": self.items_added_to_cart,
        })
        

    def simulate_session(self, url="https://demo.opencart.com/"):
        opts = Options()
        opts.add_argument("--headless")
        driver = webdriver.Chrome(options=opts)

        try:
            driver.get(url)
            self.log_action("load_home", driver.current_url)
            time.sleep(random.uniform(1, 2))

            # Scroll
            driver.execute_script("window.scrollTo(0, 800)")
            self.log_action("scroll", driver.current_url)
            time.sleep(random.uniform(1, 2))

            # Click a category
            categories = driver.find_elements(By.CSS_SELECTOR, ".nav.navbar-nav > li > a")
            if categories:
                choice = random.choice(categories)
                choice.click()
                time.sleep(random.uniform(2, 3))
                self.log_action("click_category", driver.current_url)

            # Click a product
            products = driver.find_elements(By.CSS_SELECTOR, ".product-thumb h4 a")
            if products:
                choice = random.choice(products)
                choice.click()
                time.sleep(random.uniform(2, 3))
                self.log_action("click_product", driver.current_url)

            # Add to cart
            add_btns = driver.find_elements(By.ID, "button-cart")
            if add_btns:
                add_btns[0].click()
                time.sleep(random.uniform(2, 3))
                self.log_action("add_to_cart", driver.current_url)

        finally:
            driver.quit()

    def save_logs_to_csv(self, filename="ecommerce_behavior_logs1.csv"):
        fieldnames = [
            "timestamp", "user_id", "session_id", "age_group", "tech_savviness",
            "interests", "device", "action", "page_url" , "items_added_to_cart"
        ]
        with open(filename, mode='a', newline='', encoding='utf-8') as file:
            writer = csv.DictWriter(file, fieldnames=fieldnames, quoting=csv.QUOTE_ALL)

            if file.tell() == 0:
                writer.writeheader()

            writer.writerows(self.logs)

for _ in range(10000):  # Simulate 10000 unique users
    persona = generate_random_agent()

    agent = WebBrowsingAgent(
        age_group=persona.age_group,
        tech_savviness=persona.tech_savviness,
        interests=persona.interests,
        device=persona.device
    )

agent.simulate_session()
agent.save_logs_to_csv()


In [62]:
data = pd.read_csv("ecommerce_behavior_logs1.csv")
print(data)

                            timestamp                               user_id  \
0    2025-04-13T06:40:29.889958+00:00  0a165a09-bc1c-4e51-959f-e7b6495f4fbb   
1    2025-04-13T06:40:30.974084+00:00  0a165a09-bc1c-4e51-959f-e7b6495f4fbb   
2    2025-04-13T06:43:35.670459+00:00  aa1ea0b0-6ee0-4bed-a39a-98596446cc45   
3    2025-04-13T06:43:37.574152+00:00  aa1ea0b0-6ee0-4bed-a39a-98596446cc45   
4    2025-04-13T06:43:45.467662+00:00  4cc0eba0-4391-47b1-ad88-fe970c30ac8b   
..                                ...                                   ...   
113  2025-04-13T06:53:26.640161+00:00  911ad3f4-20f7-437a-b6f8-4215f77dbc33   
114  2025-04-13T06:53:35.158876+00:00  99cdae4b-dd84-4e58-8087-3ca42a54c717   
115  2025-04-13T06:53:36.920770+00:00  99cdae4b-dd84-4e58-8087-3ca42a54c717   
116  2025-04-13T06:53:47.783443+00:00  3711120f-b5ca-4b31-b79d-7e412cfe5643   
117  2025-04-13T06:53:49.666344+00:00  3711120f-b5ca-4b31-b79d-7e412cfe5643   

                               session_id age_group

In [78]:
import sdv
import sdv.lite
from sdv.lite import SingleTablePreset


In [79]:
from sdv.lite import SingleTablePreset
from sdv.metadata import SingleTableMetadata
import pandas as pd

# Step 1: Load your dataset
data = pd.read_csv("ecommerce_behavior_logs1.csv")

# Step 2: Drop unnecessary columns
columns_to_drop = ["timestamp", "user_id", "session_id", "page_url"]
data = data.drop(columns=columns_to_drop, errors='ignore')

# Step 3: Automatically infer metadata
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data=data)
metadata.update_column(column_name="interests",sdtype="categorical")

# Step 4: Create and train the model
model = SingleTablePreset(name='FAST_ML', metadata=metadata)
model.fit(data)

# Step 5: Sample synthetic data
synthetic_data = model.sample(10000)

# Step 6: Save or view
synthetic_data.to_csv("synthetic_behavior_logs1.csv", index=False)
print(synthetic_data.head())


     age_group tech_savviness                  interests   device     action  \
0       Senior         Medium     Education,Finance,Tech   Mobile     scroll   
1        Adult         Medium  Health,Education,Shopping   Mobile     scroll   
2  Young Adult         Medium    Shopping,Health,Finance  Desktop     scroll   
3         Teen           High     Tech,Education,Finance   Mobile  load_home   
4       Senior         Medium           News,Tech,Health   Mobile     scroll   

   items_added_to_cart  
0                    0  
1                    0  
2                    0  
3                    0  
4                    0  


C:\Users\Mokshat p shah\AppData\Local\Programs\Python\Python312\Lib\site-packages\sdv\lite\single_table.py:52: FutureWarning: The 'SingleTablePreset' is deprecated. For equivalent Fast ML functionality, please use the 'GaussianCopulaSynthesizer'.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
C:\Users\Mokshat p shah\AppData\Local\Programs\Python\Python312\Lib\site-packages\sdv\lite\single_table.py:61: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(META_DEPRECATION_MSG, FutureWarning)
C:\Users\Mokshat p shah\AppData\Local\Programs\Python\Python312\Lib\site-packages\sdv\single_table\base.py:119: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
C:\Users\Mokshat p shah\AppData\Local\Programs\Python\Python312\Lib\site-packages\sdv\single_table\base.py:104: UserWarning: We strongly recommend saving the met

In [84]:
synthetic_data


,age_group,tech_savviness,interests,device,action,items_added_to_cart
0,Senior,Medium,"Education,Finance,Tech",Mobile,scroll,0
1,Adult,Medium,"Health,Education,Shopping",Mobile,scroll,0
2,Young Adult,Medium,"Shopping,Health,Finance",Desktop,scroll,0
3,Teen,High,"Tech,Education,Finance",Mobile,load_home,0
4,Senior,Medium,"News,Tech,Health",Mobile,scroll,0
...,...,...,...,...,...,...
9995,Senior,Low,"Education,News,Tech",Mobile,scroll,0
9996,Senior,Low,"Finance,Shopping,Education",Desktop,scroll,0
9997,Senior,High,"Gaming,Health,News",Mobile,scroll,0
9998,Senior,Low,"Finance,Shopping,Education",Mobile,load_home,0
